# Gandakan Kebaikan Contest Analysis

This notebook supports winner selection for the Tropicana Twister Gandakan Kebaikan contest.

Judging goals:

- Select 4 main winners based on heartfelt journal content only.
- Select 2 additional main winners based on journal content plus accumulated points.
- Select 200 additional winners based solely on points, excluding the 6 main winners.


In [5]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 160)

DATA_FILE = Path('journal-report-all-20260426.xlsx')
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

DATA_FILE.exists(), OUTPUT_DIR.resolve()


(True, WindowsPath('C:/Users/user/Documents/contest-analysis/outputs'))

## 1. Inspect workbook structure

Run this first to confirm sheet names, columns, and row counts.


In [2]:
xl = pd.ExcelFile(DATA_FILE)
xl.sheet_names


['Overview', 'Contributors Summary', 'Distribution', 'Raw Journals']

In [6]:
for sheet in xl.sheet_names:
    sample = pd.read_excel(DATA_FILE, sheet_name=sheet, nrows=5)
    print(f'\n=== {sheet} ===')
    print(f'Columns: {list(sample.columns)}')
    display(sample.head())



=== Overview ===
Columns: ['Report', 'Journal Export', 'Unnamed: 2']


,Report,Journal Export,Unnamed: 2
0,Generated At (GMT+8),2026-04-26 17:39:24 GMT+8,NaN
1,Selected Date Range,All time,NaN
2,All-Time Context,All time,NaN
3,NaN,NaN,NaN
4,Metric,Selected Range,All Time



=== Contributors Summary ===
Columns: ['participant_name', 'manychat_id', 'selected_range_journals', 'all_time_journals', 'selected_range_pledges', 'all_time_pledges', 'selected_range_oranges', 'all_time_oranges', 'first_journal_at_selected_range_gmt8', 'last_journal_at_selected_range_gmt8', 'last_activity_at_gmt8', 'subscribed_at_gmt8']


,participant_name,manychat_id,selected_range_journals,all_time_journals,selected_range_pledges,all_time_pledges,selected_range_oranges,all_time_oranges,first_journal_at_selected_range_gmt8,last_journal_at_selected_range_gmt8,last_activity_at_gmt8,subscribed_at_gmt8
0,Mega,913363032,10608,10608,52,52,106340,106340,2026-02-24 22:47:21 GMT+8,2026-04-20 14:01:25 GMT+8,2026-02-24 21:37:40 GMT+8,2026-02-24 21:37:05 GMT+8
1,Vincent Ling,996086508,8752,8752,30,30,87670,87670,2026-03-15 16:31:19 GMT+8,2026-04-17 14:49:19 GMT+8,2026-03-15 16:29:57 GMT+8,2026-03-15 16:27:17 GMT+8
2,Adam Ling,213769744,5975,5975,36,36,59930,59930,2026-02-25 13:41:21 GMT+8,2026-04-09 08:00:35 GMT+8,2026-02-25 13:40:36 GMT+8,2026-02-25 11:14:46 GMT+8
3,Iris Yu,1354271324,4068,4068,28,28,40820,40820,2026-02-24 21:40:59 GMT+8,2026-04-17 14:51:30 GMT+8,2026-03-08 14:46:24 GMT+8,2026-02-24 21:31:16 GMT+8
4,Nik Azuan Nik Azlan,749230196,1125,1125,30,30,11400,11400,2026-02-18 15:03:36 GMT+8,2026-04-11 19:13:50 GMT+8,2026-03-29 23:06:03 GMT+8,2026-02-16 16:08:39 GMT+8



=== Distribution ===
Columns: ['bucket', 'participant_count_selected_range', 'participant_pct_selected_range', 'journal_count_selected_range', 'journal_pct_selected_range', 'participant_count_all_time', 'participant_pct_all_time', 'journal_count_all_time', 'journal_pct_all_time']


,bucket,participant_count_selected_range,participant_pct_selected_range,journal_count_selected_range,journal_pct_selected_range,participant_count_all_time,participant_pct_all_time,journal_count_all_time,journal_pct_all_time
0,0,205,11.02,0,0.00,205,11.02,0,0.00
1,1-9,1580,84.95,2010,5.35,1580,84.95,2010,5.35
2,10-49,52,2.80,1080,2.88,52,2.80,1080,2.88
3,50-99,5,0.27,359,0.96,5,0.27,359,0.96
4,100-499,11,0.59,2404,6.40,11,0.59,2404,6.40



=== Raw Journals ===
Columns: ['journal_id', 'created_at_gmt8', 'participant_name', 'manychat_id', 'journal_text', 'participant_total_journals_selected_range', 'participant_total_journals_all_time', 'last_activity_at_gmt8']


,journal_id,created_at_gmt8,participant_name,manychat_id,journal_text,participant_total_journals_selected_range,participant_total_journals_all_time,last_activity_at_gmt8
0,37538,2026-04-22 00:00:22 GMT+8,Dhiea Atierah,26479135898404771,"“Saya kongsi Tropicana Twister waktu rehat, terus semua rasa segar!”",108,108,2026-04-19 10:40:32 GMT+8
1,37537,2026-04-21 23:59:35 GMT+8,Mohd Ariff,26530899793238084,"Saya ada satu kisah kebaikan saya nak diceritakan , kisah ini saya tak cerita dekat sesiapa pon atas sebab taknak tunjuk riak dan takbur. Kisah ni bermula w...",1,1,2026-04-08 07:31:59 GMT+8
2,37536,2026-04-21 23:59:34 GMT+8,Dhiea Atierah,26479135898404771,"“Saya bagi Tropicana Twister pada rakan, eratkan persahabatan kami!” 😄👍",108,108,2026-04-19 10:40:32 GMT+8
3,37535,2026-04-21 23:59:16 GMT+8,Dhiea Atierah,26479135898404771,"“Saya share Tropicana Twister, momen biasa jadi luar biasa!”",108,108,2026-04-19 10:40:32 GMT+8
4,37534,2026-04-21 23:58:57 GMT+8,Dhiea Atierah,26479135898404771,"“Saya belanja Tropicana Twister, hilang dahaga dan tambah kawan!”",108,108,2026-04-19 10:40:32 GMT+8


## 2. Load journals and participant summary

Use `Raw Journals` as the entry-level dataset and `Contributors Summary` as the participant-level points dataset.

In [7]:
journals = pd.read_excel(DATA_FILE, sheet_name='Raw Journals')
contributors = pd.read_excel(DATA_FILE, sheet_name='Contributors Summary')

print('journals:', journals.shape)
print('contributors:', contributors.shape)

display(journals.head())
display(contributors.head())

journals: (37538, 8)
contributors: (1860, 12)


,journal_id,created_at_gmt8,participant_name,manychat_id,journal_text,participant_total_journals_selected_range,participant_total_journals_all_time,last_activity_at_gmt8
0,37538,2026-04-22 00:00:22 GMT+8,Dhiea Atierah,26479135898404771,"“Saya kongsi Tropicana Twister waktu rehat, terus semua rasa segar!”",108,108,2026-04-19 10:40:32 GMT+8
1,37537,2026-04-21 23:59:35 GMT+8,Mohd Ariff,26530899793238084,"Saya ada satu kisah kebaikan saya nak diceritakan , kisah ini saya tak cerita dekat sesiapa pon atas sebab taknak tunjuk riak dan takbur. Kisah ni bermula w...",1,1,2026-04-08 07:31:59 GMT+8
2,37536,2026-04-21 23:59:34 GMT+8,Dhiea Atierah,26479135898404771,"“Saya bagi Tropicana Twister pada rakan, eratkan persahabatan kami!” 😄👍",108,108,2026-04-19 10:40:32 GMT+8
3,37535,2026-04-21 23:59:16 GMT+8,Dhiea Atierah,26479135898404771,"“Saya share Tropicana Twister, momen biasa jadi luar biasa!”",108,108,2026-04-19 10:40:32 GMT+8
4,37534,2026-04-21 23:58:57 GMT+8,Dhiea Atierah,26479135898404771,"“Saya belanja Tropicana Twister, hilang dahaga dan tambah kawan!”",108,108,2026-04-19 10:40:32 GMT+8


,participant_name,manychat_id,selected_range_journals,all_time_journals,selected_range_pledges,all_time_pledges,selected_range_oranges,all_time_oranges,first_journal_at_selected_range_gmt8,last_journal_at_selected_range_gmt8,last_activity_at_gmt8,subscribed_at_gmt8
0,Mega,913363032,10608,10608,52,52,106340,106340,2026-02-24 22:47:21 GMT+8,2026-04-20 14:01:25 GMT+8,2026-02-24 21:37:40 GMT+8,2026-02-24 21:37:05 GMT+8
1,Vincent Ling,996086508,8752,8752,30,30,87670,87670,2026-03-15 16:31:19 GMT+8,2026-04-17 14:49:19 GMT+8,2026-03-15 16:29:57 GMT+8,2026-03-15 16:27:17 GMT+8
2,Adam Ling,213769744,5975,5975,36,36,59930,59930,2026-02-25 13:41:21 GMT+8,2026-04-09 08:00:35 GMT+8,2026-02-25 13:40:36 GMT+8,2026-02-25 11:14:46 GMT+8
3,Iris Yu,1354271324,4068,4068,28,28,40820,40820,2026-02-24 21:40:59 GMT+8,2026-04-17 14:51:30 GMT+8,2026-03-08 14:46:24 GMT+8,2026-02-24 21:31:16 GMT+8
4,Nik Azuan Nik Azlan,749230196,1125,1125,30,30,11400,11400,2026-02-18 15:03:36 GMT+8,2026-04-11 19:13:50 GMT+8,2026-03-29 23:06:03 GMT+8,2026-02-16 16:08:39 GMT+8


## 3. Merge journals with points

This creates one analysis table where each journal entry includes the participant's total points (`all_time_oranges`).

In [9]:
ID_COL = 'manychat_id'
NAME_COL = 'participant_name'
JOURNAL_COL = 'journal_text'
POINTS_COL = 'all_time_oranges'

participant_cols = [
    ID_COL,
    POINTS_COL,
    'all_time_journals',
    'all_time_pledges',
    'subscribed_at_gmt8',
]

analysis_df = journals.merge(
    contributors[participant_cols],
    on=ID_COL,
    how='left',
    validate='many_to_one',
)

analysis_df['journal_text_clean'] = analysis_df[JOURNAL_COL].fillna('').astype(str).str.strip()
analysis_df['journal_text_length'] = analysis_df['journal_text_clean'].str.len()
analysis_df[POINTS_COL] = pd.to_numeric(analysis_df[POINTS_COL], errors='coerce').fillna(0)
analysis_df['all_time_journals'] = pd.to_numeric(analysis_df['all_time_journals'], errors='coerce').fillna(0)

print(analysis_df.shape)
analysis_df[[
    'journal_id',
    'created_at_gmt8',
    NAME_COL,
    ID_COL,
    JOURNAL_COL,
    'journal_text_length',
    POINTS_COL,
    'all_time_journals',
]].head()

(37538, 14)


,journal_id,created_at_gmt8,participant_name,manychat_id,journal_text,journal_text_length,all_time_oranges,all_time_journals
0,37538,2026-04-22 00:00:22 GMT+8,Dhiea Atierah,26479135898404771,"“Saya kongsi Tropicana Twister waktu rehat, terus semua rasa segar!”",68,1095,108
1,37537,2026-04-21 23:59:35 GMT+8,Mohd Ariff,26530899793238084,"Saya ada satu kisah kebaikan saya nak diceritakan , kisah ini saya tak cerita dekat sesiapa pon atas sebab taknak tunjuk riak dan takbur. Kisah ni bermula w...",1712,20,1
2,37536,2026-04-21 23:59:34 GMT+8,Dhiea Atierah,26479135898404771,"“Saya bagi Tropicana Twister pada rakan, eratkan persahabatan kami!” 😄👍",71,1095,108
3,37535,2026-04-21 23:59:16 GMT+8,Dhiea Atierah,26479135898404771,"“Saya share Tropicana Twister, momen biasa jadi luar biasa!”",60,1095,108
4,37534,2026-04-21 23:58:57 GMT+8,Dhiea Atierah,26479135898404771,"“Saya belanja Tropicana Twister, hilang dahaga dan tambah kawan!”",65,1095,108


## 4. Create judging exports

These files support manual judging:

- `journal_review_scoring_template.xlsx`: all non-empty journal entries with points included for audit.
- `content_only_review_template.xlsx`: same entries without points, for selecting the top 4 based only on heartfelt content.
- `participant_points_ranking.xlsx`: participant-level ranking by points.

In [10]:
eligible_journals = analysis_df[analysis_df['journal_text_length'] > 0].copy()

review_cols = [
    'journal_id',
    'created_at_gmt8',
    NAME_COL,
    ID_COL,
    JOURNAL_COL,
    'journal_text_length',
    POINTS_COL,
    'all_time_journals',
    'all_time_pledges',
    'last_activity_at_gmt8',
]

review = eligible_journals[review_cols].copy()
review['content_score_1_to_10'] = ''
review['heartfelt_score_1_to_10'] = ''
review['originality_score_1_to_10'] = ''
review['theme_fit_score_1_to_10'] = ''
review['judge_notes'] = ''
review['judge_decision'] = ''

content_only_cols = [
    'journal_id',
    'created_at_gmt8',
    NAME_COL,
    ID_COL,
    JOURNAL_COL,
    'journal_text_length',
    'content_score_1_to_10',
    'heartfelt_score_1_to_10',
    'originality_score_1_to_10',
    'theme_fit_score_1_to_10',
    'judge_notes',
    'judge_decision',
]

participant_points = contributors.copy()
participant_points[POINTS_COL] = pd.to_numeric(participant_points[POINTS_COL], errors='coerce').fillna(0)
participant_points = participant_points.sort_values(POINTS_COL, ascending=False)

review_path = OUTPUT_DIR / 'journal_review_scoring_template.xlsx'
content_only_path = OUTPUT_DIR / 'content_only_review_template.xlsx'
points_ranking_path = OUTPUT_DIR / 'participant_points_ranking.xlsx'

review.to_excel(review_path, index=False)
review[content_only_cols].to_excel(content_only_path, index=False)
participant_points.to_excel(points_ranking_path, index=False)

print('Eligible journal entries:', len(eligible_journals))
print('Unique participants with journals:', eligible_journals[ID_COL].nunique())
print('Saved:', review_path)
print('Saved:', content_only_path)
print('Saved:', points_ranking_path)

Eligible journal entries: 37538
Unique participants with journals: 1655
Saved: outputs\journal_review_scoring_template.xlsx
Saved: outputs\content_only_review_template.xlsx
Saved: outputs\participant_points_ranking.xlsx


## 5. Select additional 200 winners by points

After the 6 main winners are confirmed, enter their `manychat_id` values in `MAIN_WINNER_IDS`. This exports the next 200 highest-point participants, excluding the main winners.

In [12]:
MAIN_WINNER_IDS = []

additional_200 = (
    participant_points[~participant_points[ID_COL].isin(MAIN_WINNER_IDS)]
    .sort_values(POINTS_COL, ascending=False)
    .head(200)
    .copy()
)

additional_path = OUTPUT_DIR / 'additional_200_winners_by_points.xlsx'
additional_200.to_excel(additional_path, index=False)

print('Saved:', additional_path)
additional_200[[NAME_COL, ID_COL, POINTS_COL, 'all_time_journals', 'all_time_pledges']].head(20)

Saved: outputs\additional_200_winners_by_points.xlsx


,participant_name,manychat_id,all_time_oranges,all_time_journals,all_time_pledges
0,Mega,913363032,106340,10608,52
1,Vincent Ling,996086508,87670,8752,30
2,Adam Ling,213769744,59930,5975,36
3,Iris Yu,1354271324,40820,4068,28
4,Nik Azuan Nik Azlan,749230196,11400,1125,30
5,Beckham Wong,1411599678,6260,622,8
6,Swee Peu Yu,27007450552189766,5375,535,5
7,Komando Fairus,236510941,3815,369,25
8,Ting Ngik Sieng,1643097584,3695,362,15
9,Ngik,26145998711700649,3120,307,10


In [ ]:
## 6. Build a content shortlist

This section creates practical shortlist files for human review. The scoring here is only a triage aid, not the final judging result.

The heuristic prioritizes entries that are longer, personal, story-like, emotional, and connected to kindness, while reducing repeated/template-like entries.

In [13]:
import re

personal_terms = [
    'saya', 'aku', 'kami', 'keluarga', 'ibu', 'mak', 'mama', 'ayah', 'bapa', 'papa',
    'anak', 'isteri', 'suami', 'adik', 'abang', 'kakak', 'nenek', 'datuk', 'rakan',
    'kawan', 'jiran', 'my', 'i ', 'me ', 'we ', 'family', 'mother', 'father', 'friend',
]

kindness_terms = [
    'baik', 'kebaikan', 'bantu', 'membantu', 'tolong', 'menolong', 'hulur', 'kongsi',
    'berkongsi', 'sedekah', 'ikhlas', 'prihatin', 'kasih', 'sayang', 'senyum', 'gembira',
    'terima kasih', 'appreciate', 'kindness', 'help', 'helped', 'share', 'shared', 'care',
]

emotion_terms = [
    'sedih', 'terharu', 'menangis', 'syukur', 'bersyukur', 'gembira', 'bahagia', 'susah',
    'sukar', 'cabaran', 'dugaan', 'letih', 'penat', 'ikhlas', 'tersentuh', 'harapan',
    'sad', 'touched', 'grateful', 'thankful', 'happy', 'struggle', 'challenge', 'hope',
]

story_terms = [
    'pada suatu', 'hari itu', 'ketika', 'semasa', 'waktu', 'masa tu', 'bermula', 'kisah',
    'cerita', 'pengalaman', 'akhirnya', 'selepas', 'sebelum', 'then', 'when', 'after',
    'before', 'story', 'experience',
]

promo_terms = [
    'tropicana twister', 'twister', 'rasa segar', 'hilang dahaga', 'orange', 'oren',
]

def normalize_text(value):
    text = str(value).lower().strip()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s]', '', text)
    return text

def count_terms(text, terms):
    normalized = f' {normalize_text(text)} '
    return sum(1 for term in terms if f' {normalize_text(term)} ' in normalized)

shortlist_df = eligible_journals.copy()
shortlist_df['normalized_journal_text'] = shortlist_df[JOURNAL_COL].map(normalize_text)
shortlist_df['duplicate_text_count'] = shortlist_df.groupby('normalized_journal_text')['journal_id'].transform('count')
shortlist_df['participant_entry_rank_by_length'] = shortlist_df.groupby(ID_COL)['journal_text_length'].rank(method='first', ascending=False)

shortlist_df['personal_term_count'] = shortlist_df[JOURNAL_COL].map(lambda x: count_terms(x, personal_terms))
shortlist_df['kindness_term_count'] = shortlist_df[JOURNAL_COL].map(lambda x: count_terms(x, kindness_terms))
shortlist_df['emotion_term_count'] = shortlist_df[JOURNAL_COL].map(lambda x: count_terms(x, emotion_terms))
shortlist_df['story_term_count'] = shortlist_df[JOURNAL_COL].map(lambda x: count_terms(x, story_terms))
shortlist_df['promo_term_count'] = shortlist_df[JOURNAL_COL].map(lambda x: count_terms(x, promo_terms))

shortlist_df['length_score'] = np.clip(shortlist_df['journal_text_length'] / 1200, 0, 1) * 30
shortlist_df['personal_score'] = np.clip(shortlist_df['personal_term_count'], 0, 5) * 5
shortlist_df['kindness_score'] = np.clip(shortlist_df['kindness_term_count'], 0, 5) * 5
shortlist_df['emotion_score'] = np.clip(shortlist_df['emotion_term_count'], 0, 5) * 4
shortlist_df['story_score'] = np.clip(shortlist_df['story_term_count'], 0, 5) * 4
shortlist_df['duplicate_penalty'] = np.where(shortlist_df['duplicate_text_count'] > 1, 25, 0)
shortlist_df['short_generic_promo_penalty'] = np.where(
    (shortlist_df['journal_text_length'] < 120) & (shortlist_df['promo_term_count'] > 0),
    20,
    0,
)

shortlist_df['heuristic_content_score'] = (
    shortlist_df['length_score']
    + shortlist_df['personal_score']
    + shortlist_df['kindness_score']
    + shortlist_df['emotion_score']
    + shortlist_df['story_score']
    - shortlist_df['duplicate_penalty']
    - shortlist_df['short_generic_promo_penalty']
).round(2)

shortlist_cols = [
    'journal_id',
    'created_at_gmt8',
    NAME_COL,
    ID_COL,
    JOURNAL_COL,
    'journal_text_length',
    'heuristic_content_score',
    'personal_term_count',
    'kindness_term_count',
    'emotion_term_count',
    'story_term_count',
    'duplicate_text_count',
    POINTS_COL,
    'all_time_journals',
    'all_time_pledges',
]

content_shortlist_top_300 = (
    shortlist_df
    .sort_values(['heuristic_content_score', 'journal_text_length'], ascending=[False, False])
    .head(300)[shortlist_cols]
    .copy()
)

participant_best_content = (
    shortlist_df
    .sort_values(['heuristic_content_score', 'journal_text_length'], ascending=[False, False])
    .drop_duplicates(ID_COL)
    .head(300)[shortlist_cols]
    .copy()
)

long_story_review = (
    shortlist_df[shortlist_df['journal_text_length'] >= 500]
    .sort_values(['heuristic_content_score', 'journal_text_length'], ascending=[False, False])
    .head(300)[shortlist_cols]
    .copy()
)

content_shortlist_path = OUTPUT_DIR / 'content_shortlist_top_300.xlsx'
participant_best_path = OUTPUT_DIR / 'participant_best_content_shortlist_top_300.xlsx'
long_story_path = OUTPUT_DIR / 'long_story_review_top_300.xlsx'

content_shortlist_top_300.to_excel(content_shortlist_path, index=False)
participant_best_content.to_excel(participant_best_path, index=False)
long_story_review.to_excel(long_story_path, index=False)

print('Saved:', content_shortlist_path)
print('Saved:', participant_best_path)
print('Saved:', long_story_path)
print('Top shortlist entries:', len(content_shortlist_top_300))
print('Participant-best entries:', len(participant_best_content))
print('Long-story entries:', len(long_story_review))

content_shortlist_top_300.head(20)

Saved: outputs\content_shortlist_top_300.xlsx
Saved: outputs\participant_best_content_shortlist_top_300.xlsx
Saved: outputs\long_story_review_top_300.xlsx
Top shortlist entries: 300
Participant-best entries: 300
Long-story entries: 77


,journal_id,created_at_gmt8,participant_name,manychat_id,journal_text,journal_text_length,heuristic_content_score,personal_term_count,kindness_term_count,emotion_term_count,story_term_count,duplicate_text_count,all_time_oranges,all_time_journals,all_time_pledges
1,37537,2026-04-21 23:59:35 GMT+8,Mohd Ariff,26530899793238084,"Saya ada satu kisah kebaikan saya nak diceritakan , kisah ini saya tak cerita dekat sesiapa pon atas sebab taknak tunjuk riak dan takbur. Kisah ni bermula w...",1712,95.00,7,4,0,5,1,20,1,2
32577,4961,2026-03-03 14:16:54 GMT+8,Jen,550611024,"Hari ini, saya cuba menyebarkan kebaikan dalam cara-cara kecil yang bermakna 💖✨. Pagi tadi, saya bantu jiran angkat barang dari kereta ke rumahnya 🏡. Lepas ...",686,75.15,5,6,2,0,1,15,1,1
31524,6014,2026-03-06 15:33:20 GMT+8,LaiLa La,34057189250591023,Satu cerita yang ingin saya kongsikan tentang Tropicana Malaysia. Pada bulan Ramadan tahun 2021 di pasar raya terdekat telah mengadakan promosi air Tropican...,1356,72.00,3,3,2,1,1,85,7,3
34691,2847,2026-02-27 23:45:00 GMT+8,Nur Syifa Syairah,26178226831812452,Saya belanja ibu dan adik-adik makan serta membeli barang keperluan rumah dengan hati yang penuh rasa tanggungjawab . Ibu adalah seorang ibu tunggal sejak a...,794,71.85,5,3,2,1,1,185,13,11
37111,427,2026-02-21 22:06:56 GMT+8,Sheikh,26455826147348071,"Sejak ayah meninggal dunia pada 2010, saya belajar menjadi lebih kuat untuk mak. Awal Ramadan, saya ziarah pusara ayah sendirian dan berdoa agar dapat terus...",469,69.72,5,6,2,0,1,50,4,2
32578,4960,2026-03-03 14:14:30 GMT+8,Zahra,33962513566727192,"Saya bantu memegang bayi sementara ibunya membancuh susu. Ketika itu saya berada di klinik kesihatan, saya melihat seorang ibu muda kelihatan gelisah apabil...",1026,67.65,2,4,2,1,1,140,9,10
31523,6015,2026-03-06 15:47:10 GMT+8,LaiLa La,34057189250591023,"Beberapa hari lepas, ibu minta untuk belikan beberapa botol air Tropicana Twister Oren untuk dikongsikan bersama jemaah surau dan kawan-kawannya pada Ramada...",957,66.92,3,4,0,2,1,85,7,3
19850,17688,2026-03-19 03:05:49 GMT+8,Hawatif Hanim,347684135,Ini kisah kebaikan yang telah kulakukan pada awal bulan berpuasa. Cuaca di siang hari semestinya sangat terik dan bahang. Kami sekeluarga berpuasa dengan pe...,662,66.55,4,2,2,3,1,25,2,1
35883,1655,2026-02-27 10:17:25 GMT+8,Aini Nazlifa,342377315,"Zaman kanak-kanak yang paling saya ingati adalah ketika bersama seorang figura yang dipanggil 'opah'. Pada waktu kecil, saya membantu opah saya menjual ais...",1147,65.68,3,2,1,2,1,20,1,2
36983,555,2026-02-23 18:35:29 GMT+8,Kei Chin,373359919,"Pada suatu petang di bulan Ramadan, saya ternampak seorang pakcik warga emas duduk sendirian di luar sebuah kedai kecil sementara orang lain sibuk membeli j...",901,63.53,1,4,2,2,1,1475,144,7


## 7. Review guidance for selecting the top 6

Recommended order:

1. Open `content_only_review_template.xlsx` or `participant_best_content_shortlist_top_300.xlsx` without using points to select the strongest heartfelt stories.
2. Choose the top 4 based on content only.
3. Use `journal_review_scoring_template.xlsx` or the shortlist with points visible to choose the next 2 using both content and points.
4. Enter all 6 selected `manychat_id` values into `MAIN_WINNER_IDS` in Section 5, then rerun Section 5 to regenerate the additional 200 winners.